#**Tahapan Pelatihan dan Evaluasi Model**

## **Langkah 9: Model Training**

####**9.1 Setup Folder Binary OvR**

In [ ]:
# Root folder binary - semua output disimpan di sini
BINARY_BASE_DIR        = '/content/drive/My Drive/Framework/CICIoT2023/Binary/'
BINARY_CHECKPOINTS_DIR = '/content/drive/My Drive/Framework/CICIoT2023/Binary/Checkpoints/'
BINARY_MODELS_DIR      = '/content/drive/My Drive/Framework/CICIoT2023/Binary/Models/'
BINARY_RESULTS_DIR     = '/content/drive/My Drive/Framework/CICIoT2023/Binary/Results/'
BINARY_CM_DIR          = '/content/drive/My Drive/Framework/CICIoT2023/Binary/Results/Confusion_Matrices/'
BINARY_VIZ_DIR         = '/content/drive/My Drive/Framework/CICIoT2023/Binary/Results/Visualizations/'

# Sub-folder per classifier di dalam Models
BINARY_RF_DIR    = os.path.join(BINARY_MODELS_DIR, 'RandomForest/')
BINARY_XGB_DIR   = os.path.join(BINARY_MODELS_DIR, 'XGBoost/')
BINARY_LGB_DIR   = os.path.join(BINARY_MODELS_DIR, 'LightGBM/')
BINARY_DNN_DIR   = os.path.join(BINARY_MODELS_DIR, 'DNN/')

for d in [
    BINARY_BASE_DIR, BINARY_CHECKPOINTS_DIR,
    BINARY_MODELS_DIR, BINARY_RESULTS_DIR,
    BINARY_CM_DIR, BINARY_VIZ_DIR,
    BINARY_RF_DIR, BINARY_XGB_DIR,
    BINARY_LGB_DIR, BINARY_DNN_DIR
]:
    os.makedirs(d, exist_ok=True)

print("Folder Binary OvR berhasil dibuat:")
print(f"  Base      : {BINARY_BASE_DIR}")
print(f"  Models    : {BINARY_MODELS_DIR}")
print(f"  Results   : {BINARY_RESULTS_DIR}")
print(f"  CM        : {BINARY_CM_DIR}")
print(f"  Viz       : {BINARY_VIZ_DIR}")

Folder Binary OvR berhasil dibuat:
  Base      : /content/drive/My Drive/Framework/CICIoT2023/Binary/
  Models    : /content/drive/My Drive/Framework/CICIoT2023/Binary/Models/
  Results   : /content/drive/My Drive/Framework/CICIoT2023/Binary/Results/
  CM        : /content/drive/My Drive/Framework/CICIoT2023/Binary/Results/Confusion_Matrices/
  Viz       : /content/drive/My Drive/Framework/CICIoT2023/Binary/Results/Visualizations/


####**9.2 Load Dataset Final**

In [ ]:
def ram_usage():
    mem = psutil.virtual_memory()
    return (mem.total - mem.available) / 1024**3, mem.total / 1024**3

def print_ram(label=""):
    used, total = ram_usage()
    print(f"  RAM [{label}]: {used:.2f}/{total:.2f} GB")

def load_arr(var_name, path, dtype):
    if var_name in globals() and globals()[var_name] is not None:
        arr = globals()[var_name]
        print(f"  {var_name:30s} : memory  shape={arr.shape}")
        return arr
    print(f"  {var_name:30s} : loading dari disk...")
    arr = np.load(path, allow_pickle=False).astype(dtype)
    print(f"  {var_name:30s} : shape={arr.shape}")
    return arr

X_train_final = load_arr(
    'X_train_final',
    os.path.join(FINAL_DIR, 'X_train_final.npy'),
    np.float32
)
y_train_final = load_arr(
    'y_train_final',
    os.path.join(FINAL_DIR, 'y_train_final.npy'),
    np.int32
)
X_test = load_arr(
    'X_test',
    os.path.join(FINAL_DIR, 'X_test_final.npy'),
    np.float32
)
y_test = load_arr(
    'y_test',
    os.path.join(FINAL_DIR, 'y_test_final.npy'),
    np.int32
)

N_CLASSES  = len(LABEL_NAMES)
N_FEATURES = X_train_final.shape[1]
N_TRAIN    = len(X_train_final)

print(f"\nX_train_final : {X_train_final.shape}  "
      f"({X_train_final.nbytes/1024**3:.2f} GB)")
print(f"X_test        : {X_test.shape}")
print(f"n_classes     : {N_CLASSES}")
print(f"n_features    : {N_FEATURES}")
print_ram("setelah load")

  X_train_final                  : loading dari disk...
  X_train_final                  : shape=(20302105, 39)
  y_train_final                  : loading dari disk...
  y_train_final                  : shape=(20302105,)
  X_test                         : loading dari disk...
  X_test                         : shape=(4279721, 39)
  y_test                         : loading dari disk...
  y_test                         : shape=(4279721,)

X_train_final : (20302105, 39)  (2.95 GB)
X_test        : (4279721, 39)
n_classes     : 8
n_features    : 39
  RAM [setelah load]: 6.73/50.99 GB


####**9.3 Setup Helper Functions**

In [ ]:
def print_binary_cm(y_true_bin, y_pred_bin, class_name):
    """Cetak confusion matrix binary dalam format TN/FP/FN/TP."""
    cm = confusion_matrix(y_true_bin, y_pred_bin)
    tn, fp, fn, tp = cm.ravel()
    total    = tn + fp + fn + tp
    tpr      = tp / (tp + fn) if (tp + fn) > 0 else 0.0   # recall / sensitivity
    tnr      = tn / (tn + fp) if (tn + fp) > 0 else 0.0   # specificity
    fpr      = fp / (fp + tn) if (fp + tn) > 0 else 0.0   # false positive rate
    fnr      = fn / (fn + tp) if (fn + tp) > 0 else 0.0   # false negative rate
    print(f"\n  Confusion Matrix [{class_name}]:")
    print(f"    TN={tn:>10,} | FP={fp:>10,}")
    print(f"    FN={fn:>10,} | TP={tp:>10,}")
    print(f"    Total test samples : {total:,}")
    print(f"    TPR (Recall/Sens.) : {tpr:.6f}")
    print(f"    TNR (Specificity)  : {tnr:.6f}")
    print(f"    FPR                : {fpr:.6f}")
    print(f"    FNR                : {fnr:.6f}")
    return {'tn': int(tn), 'fp': int(fp), 'fn': int(fn), 'tp': int(tp),
            'tpr': float(tpr), 'tnr': float(tnr),
            'fpr': float(fpr), 'fnr': float(fnr)}

def compute_binary_metrics(y_true_bin, y_pred_bin, y_prob_bin, class_name):
    """Hitung metrics binary lengkap."""
    acc  = float(accuracy_score(y_true_bin, y_pred_bin))
    prec = float(precision_score(y_true_bin, y_pred_bin, zero_division=0))
    rec  = float(recall_score(y_true_bin, y_pred_bin, zero_division=0))
    f1   = float(f1_score(y_true_bin, y_pred_bin, zero_division=0))
    mcc  = float(matthews_corrcoef(y_true_bin, y_pred_bin))

    roc_auc = float(roc_auc_score(y_true_bin, y_prob_bin)) \
        if y_prob_bin is not None else None
    pr_auc  = float(average_precision_score(y_true_bin, y_prob_bin)) \
        if y_prob_bin is not None else None

    cm_dict = print_binary_cm(y_true_bin, y_pred_bin, class_name)

    print(f"\n  Metrics [{class_name}]:")
    print(f"    Accuracy   : {acc:.6f}")
    print(f"    Precision  : {prec:.6f}")
    print(f"    Recall     : {rec:.6f}")
    print(f"    F1-Score   : {f1:.6f}")
    print(f"    MCC        : {mcc:.6f}")
    if roc_auc is not None:
        print(f"    ROC-AUC    : {roc_auc:.6f}")
    if pr_auc is not None:
        print(f"    PR-AUC     : {pr_auc:.6f}")

    return {
        'class_name' : class_name,
        'accuracy'   : acc,
        'precision'  : prec,
        'recall'     : rec,
        'f1'         : f1,
        'mcc'        : mcc,
        'roc_auc'    : roc_auc,
        'pr_auc'     : pr_auc,
        'cm'         : cm_dict
    }

def compute_ovr_summary(per_class_results, clf_name):
    """Hitung macro dan weighted average dari semua kelas OvR."""
    f1_list   = [r['f1']        for r in per_class_results.values()]
    prec_list = [r['precision'] for r in per_class_results.values()]
    rec_list  = [r['recall']    for r in per_class_results.values()]
    mcc_list  = [r['mcc']       for r in per_class_results.values()]
    roc_list  = [r['roc_auc']   for r in per_class_results.values()
                 if r['roc_auc'] is not None]
    pr_list   = [r['pr_auc']    for r in per_class_results.values()
                 if r['pr_auc'] is not None]

    summary = {
        'macro_precision' : float(np.mean(prec_list)),
        'macro_recall'    : float(np.mean(rec_list)),
        'macro_f1'        : float(np.mean(f1_list)),
        'macro_mcc'       : float(np.mean(mcc_list)),
        'macro_roc_auc'   : float(np.mean(roc_list)) if roc_list else None,
        'macro_pr_auc'    : float(np.mean(pr_list))  if pr_list  else None,
    }

    print(f"\n  {'='*50}")
    print(f"  Ringkasan OvR - {clf_name}")
    print(f"  {'='*50}")
    print(f"  Macro Precision : {summary['macro_precision']:.6f}")
    print(f"  Macro Recall    : {summary['macro_recall']:.6f}")
    print(f"  Macro F1        : {summary['macro_f1']:.6f}")
    print(f"  Macro MCC       : {summary['macro_mcc']:.6f}")
    if summary['macro_roc_auc']:
        print(f"  Macro ROC-AUC   : {summary['macro_roc_auc']:.6f}")
    if summary['macro_pr_auc']:
        print(f"  Macro PR-AUC    : {summary['macro_pr_auc']:.6f}")

    return summary

# Dict utama untuk semua hasil
all_results_binary = {}

print("Helper functions siap.")

Helper functions siap.


####**9.4 Random Forest - Binary OvR**

In [ ]:
# Cleanup variabel tidak diperlukan
for v in ['X_dominant', 'y_dominant', 'X_moderate_augmented',
          'y_moderate_augmented', 'X_critical_augmented',
          'y_critical_augmented']:
    if v in globals():
        del globals()[v]
gc.collect()
print_ram("sebelum RF OvR")

  RAM [sebelum RF OvR]: 10.15/50.99 GB


In [ ]:
RF_PARAMS_BIN = {
    'n_estimators'      : 100,
    'max_depth'         : 20,
    'min_samples_split' : 5,
    'min_samples_leaf'  : 2,
    'max_features'      : 'sqrt',
    'bootstrap'         : True,
    'class_weight'      : None,
    'n_jobs'            : -1,
    'random_state'      : 42,
    'verbose'           : 0
}

print("Parameter RF Binary OvR:")
for k, v in RF_PARAMS_BIN.items():
    print(f"  {k:25s} : {v}")

rf_results_per_class = {}
# Matrix untuk menyimpan probabilitas kelas positif tiap kelas
y_prob_rf_matrix = np.zeros((len(y_test), N_CLASSES), dtype=np.float32)

t_rf_total = 0
for enc in range(N_CLASSES):
    class_name = LABEL_NAMES[enc]
    print(f"\n--- RF | Kelas {enc}: {class_name} ---")

    # Buat binary label
    y_train_bin = (y_train_final == enc).astype(np.int32)
    y_test_bin  = (y_test == enc).astype(np.int32)

    n_pos_train = int(y_train_bin.sum())
    n_neg_train = int((y_train_bin == 0).sum())
    n_pos_test  = int(y_test_bin.sum())
    print(f"  Train: pos={n_pos_train:,}  neg={n_neg_train:,}")
    print(f"  Test : pos={n_pos_test:,}")

    # Training
    t0 = time.time()
    rf_bin = RandomForestClassifier(**RF_PARAMS_BIN)
    rf_bin.fit(X_train_final, y_train_bin)
    t_elapsed = time.time() - t0
    t_rf_total += t_elapsed
    print(f"  Training selesai: {t_elapsed:.2f}s")

    # Prediksi
    y_pred_bin  = rf_bin.predict(X_test)
    y_prob_bin  = rf_bin.predict_proba(X_test)[:, 1].astype(np.float32)
    y_prob_rf_matrix[:, enc] = y_prob_bin

    # Metrics + CM
    res = compute_binary_metrics(y_test_bin, y_pred_bin, y_prob_bin, class_name)
    res['training_time_sec'] = float(t_elapsed)
    rf_results_per_class[class_name] = res

    # Save model per kelas
    path_m = os.path.join(BINARY_RF_DIR, f'rf_bin_{class_name.lower()}.pkl')
    with open(path_m, 'wb') as f:
        pickle.dump(rf_bin, f)

    del rf_bin, y_train_bin, y_pred_bin, y_prob_bin
    gc.collect()

# Final prediction: argmax probabilitas
y_pred_rf_ovr = np.argmax(y_prob_rf_matrix, axis=1).astype(np.int32)

# Summary OvR
rf_summary = compute_ovr_summary(rf_results_per_class, "RandomForest")

# Save predictions dan results
np.save(os.path.join(BINARY_RF_DIR, 'y_pred_rf_ovr.npy'), y_pred_rf_ovr)
np.save(os.path.join(BINARY_RF_DIR, 'y_prob_rf_matrix.npy'), y_prob_rf_matrix)

all_results_binary['RandomForest'] = {
    'per_class'         : rf_results_per_class,
    'summary'           : rf_summary,
    'total_training_sec': float(t_rf_total)
}

with open(os.path.join(BINARY_RESULTS_DIR, 'result_rf_binary.json'), 'w') as f:
    json.dump(all_results_binary['RandomForest'], f, indent=4)

print(f"\nTotal waktu RF OvR  : {t_rf_total:.2f}s ({t_rf_total/60:.2f} menit)")
print_ram("setelah RF OvR")

del y_prob_rf_matrix
gc.collect()
print("RF Binary OvR selesai.")

Parameter RF Binary OvR:
  n_estimators              : 100
  max_depth                 : 20
  min_samples_split         : 5
  min_samples_leaf          : 2
  max_features              : sqrt
  bootstrap                 : True
  class_weight              : None
  n_jobs                    : -1
  random_state              : 42
  verbose                   : 0

--- RF | Kelas 0: Benign ---
  Train: pos=1,000,000  neg=19,302,105
  Test : pos=218,760
  Training selesai: 1166.28s

  Confusion Matrix [Benign]:
    TN= 4,024,388 | FP=    36,573
    FN=    29,534 | TP=   189,226
    Total test samples : 4,279,721
    TPR (Recall/Sens.) : 0.864994
    TNR (Specificity)  : 0.990994
    FPR                : 0.009006
    FNR                : 0.135006

  Metrics [Benign]:
    Accuracy   : 0.984553
    Precision  : 0.838029
    Recall     : 0.864994
    F1-Score   : 0.851298
    MCC        : 0.843271
    ROC-AUC    : 0.996120
    PR-AUC     : 0.915910

--- RF | Kelas 1: Brute_Force ---
  Train: pos=1,

####**9.5 XGBoost - Binary OvR**

In [ ]:
XGB_PARAMS_BIN = {
    'n_estimators'      : 100,
    'max_depth'         : 10,
    'learning_rate'     : 0.1,
    'subsample'         : 0.8,
    'colsample_bytree'  : 0.8,
    'objective'         : 'binary:logistic',
    'eval_metric'       : 'logloss',
    'tree_method'       : 'hist',
    'device'            : 'cuda',
    'random_state'      : 42,
    'n_jobs'            : -1,
    'verbosity'         : 2
}

print("Parameter XGBoost Binary OvR:")
for k, v in XGB_PARAMS_BIN.items():
    print(f"  {k:25s} : {v}")

xgb_results_per_class = {}
y_prob_xgb_matrix = np.zeros((len(y_test), N_CLASSES), dtype=np.float32)

t_xgb_total = 0
for enc in range(N_CLASSES):
    class_name = LABEL_NAMES[enc]
    print(f"\n--- XGB | Kelas {enc}: {class_name} ---")

    y_train_bin = (y_train_final == enc).astype(np.int32)
    y_test_bin  = (y_test == enc).astype(np.int32)

    n_pos_train = int(y_train_bin.sum())
    n_neg_train = int((y_train_bin == 0).sum())
    n_pos_test  = int(y_test_bin.sum())
    print(f"  Train: pos={n_pos_train:,}  neg={n_neg_train:,}")
    print(f"  Test : pos={n_pos_test:,}")

    t0 = time.time()
    xgb_bin = xgb.XGBClassifier(**XGB_PARAMS_BIN)
    xgb_bin.fit(
        X_train_final, y_train_bin,
        eval_set=[(X_test, y_test_bin)],
        verbose=50
    )
    t_elapsed = time.time() - t0
    t_xgb_total += t_elapsed
    print(f"  Training selesai: {t_elapsed:.2f}s")

    y_pred_bin = xgb_bin.predict(X_test)
    y_prob_bin = xgb_bin.predict_proba(X_test)[:, 1].astype(np.float32)
    y_prob_xgb_matrix[:, enc] = y_prob_bin

    res = compute_binary_metrics(y_test_bin, y_pred_bin, y_prob_bin, class_name)
    res['training_time_sec'] = float(t_elapsed)
    xgb_results_per_class[class_name] = res

    path_m = os.path.join(BINARY_XGB_DIR, f'xgb_bin_{class_name.lower()}.json')
    xgb_bin.save_model(path_m)

    del xgb_bin, y_train_bin, y_pred_bin, y_prob_bin
    gc.collect()

y_pred_xgb_ovr = np.argmax(y_prob_xgb_matrix, axis=1).astype(np.int32)
xgb_summary    = compute_ovr_summary(xgb_results_per_class, "XGBoost")

np.save(os.path.join(BINARY_XGB_DIR, 'y_pred_xgb_ovr.npy'), y_pred_xgb_ovr)
np.save(os.path.join(BINARY_XGB_DIR, 'y_prob_xgb_matrix.npy'), y_prob_xgb_matrix)

all_results_binary['XGBoost'] = {
    'per_class'         : xgb_results_per_class,
    'summary'           : xgb_summary,
    'total_training_sec': float(t_xgb_total)
}

with open(os.path.join(BINARY_RESULTS_DIR, 'result_xgb_binary.json'), 'w') as f:
    json.dump(all_results_binary['XGBoost'], f, indent=4)

print(f"\nTotal waktu XGB OvR : {t_xgb_total:.2f}s ({t_xgb_total/60:.2f} menit)")
print_ram("setelah XGB OvR")
del y_prob_xgb_matrix
gc.collect()
print("XGBoost Binary OvR selesai.")

Parameter XGBoost Binary OvR:
  n_estimators              : 100
  max_depth                 : 10
  learning_rate             : 0.1
  subsample                 : 0.8
  colsample_bytree          : 0.8
  objective                 : binary:logistic
  eval_metric               : logloss
  tree_method               : hist
  device                    : cuda
  random_state              : 42
  n_jobs                    : -1
  verbosity                 : 2

--- XGB | Kelas 0: Benign ---
  Train: pos=1,000,000  neg=19,302,105
  Test : pos=218,760
[04:07:35] INFO: /__w/xgboost/xgboost/src/data/iterative_dmatrix.cc:56: Finished constructing the `IterativeDMatrix`: (20302105, 39, 791782095).
[04:07:37] INFO: /__w/xgboost/xgboost/src/data/iterative_dmatrix.cc:56: Finished constructing the `IterativeDMatrix`: (4279721, 39, 166909119).
[04:07:37] INFO: /__w/xgboost/xgboost/src/data/ellpack_page.cu:170: Ellpack is dense.
[04:07:37] INFO: /__w/xgboost/xgboost/src/common/cuda_dr_utils.cc:180: Driver versi

####**9.6 LightGBM - Binary OvR**

In [ ]:
LGB_PARAMS_BIN = {
    'n_estimators'      : 100,
    'max_depth'         : 10,
    'learning_rate'     : 0.1,
    'num_leaves'        : 31,
    'subsample'         : 0.8,
    'colsample_bytree'  : 0.8,
    'objective'         : 'binary',
    'class_weight'      : None,
    'device'            : 'cpu',
    'random_state'      : 42,
    'n_jobs'            : -1,
    'verbose'           : -1
}

print("Parameter LightGBM Binary OvR:")
for k, v in LGB_PARAMS_BIN.items():
    print(f"  {k:25s} : {v}")

lgb_results_per_class = {}
y_prob_lgb_matrix = np.zeros((len(y_test), N_CLASSES), dtype=np.float32)

t_lgb_total = 0
for enc in range(N_CLASSES):
    class_name = LABEL_NAMES[enc]
    print(f"\n--- LGB | Kelas {enc}: {class_name} ---")

    y_train_bin = (y_train_final == enc).astype(np.int32)
    y_test_bin  = (y_test == enc).astype(np.int32)

    n_pos_train = int(y_train_bin.sum())
    n_neg_train = int((y_train_bin == 0).sum())
    n_pos_test  = int(y_test_bin.sum())
    print(f"  Train: pos={n_pos_train:,}  neg={n_neg_train:,}")
    print(f"  Test : pos={n_pos_test:,}")

    t0 = time.time()
    lgb_bin = lgb.LGBMClassifier(**LGB_PARAMS_BIN)
    lgb_bin.fit(
        X_train_final, y_train_bin,
        eval_set=[(X_test, y_test_bin)],
        callbacks=[
            lgb.early_stopping(stopping_rounds=20, verbose=False),
            lgb.log_evaluation(period=50)
        ]
    )
    t_elapsed = time.time() - t0
    t_lgb_total += t_elapsed
    print(f"  Training selesai: {t_elapsed:.2f}s")

    y_pred_bin = lgb_bin.predict(X_test).astype(np.int32)
    y_prob_bin = lgb_bin.predict_proba(X_test)[:, 1].astype(np.float32)
    y_prob_lgb_matrix[:, enc] = y_prob_bin

    res = compute_binary_metrics(y_test_bin, y_pred_bin, y_prob_bin, class_name)
    res['training_time_sec'] = float(t_elapsed)
    lgb_results_per_class[class_name] = res

    path_m = os.path.join(BINARY_LGB_DIR, f'lgb_bin_{class_name.lower()}.txt')
    lgb_bin.booster_.save_model(path_m)

    del lgb_bin, y_train_bin, y_pred_bin, y_prob_bin
    gc.collect()

y_pred_lgb_ovr = np.argmax(y_prob_lgb_matrix, axis=1).astype(np.int32)
lgb_summary    = compute_ovr_summary(lgb_results_per_class, "LightGBM")

np.save(os.path.join(BINARY_LGB_DIR, 'y_pred_lgb_ovr.npy'), y_pred_lgb_ovr)
np.save(os.path.join(BINARY_LGB_DIR, 'y_prob_lgb_matrix.npy'), y_prob_lgb_matrix)

all_results_binary['LightGBM'] = {
    'per_class'         : lgb_results_per_class,
    'summary'           : lgb_summary,
    'total_training_sec': float(t_lgb_total)
}

with open(os.path.join(BINARY_RESULTS_DIR, 'result_lgb_binary.json'), 'w') as f:
    json.dump(all_results_binary['LightGBM'], f, indent=4)

print(f"\nTotal waktu LGB OvR : {t_lgb_total:.2f}s ({t_lgb_total/60:.2f} menit)")
print_ram("setelah LGB OvR")
del y_prob_lgb_matrix
gc.collect()
print("LightGBM Binary OvR selesai.")

Parameter LightGBM Binary OvR:
  n_estimators              : 100
  max_depth                 : 10
  learning_rate             : 0.1
  num_leaves                : 31
  subsample                 : 0.8
  colsample_bytree          : 0.8
  objective                 : binary
  class_weight              : None
  device                    : cpu
  random_state              : 42
  n_jobs                    : -1
  verbose                   : -1

--- LGB | Kelas 0: Benign ---
  Train: pos=1,000,000  neg=19,302,105
  Test : pos=218,760
[50]	valid_0's binary_logloss: 0.037824
[100]	valid_0's binary_logloss: 0.0359028
  Training selesai: 82.83s

  Confusion Matrix [Benign]:
    TN= 4,021,726 | FP=    39,235
    FN=    29,708 | TP=   189,052
    Total test samples : 4,279,721
    TPR (Recall/Sens.) : 0.864198
    TNR (Specificity)  : 0.990338
    FPR                : 0.009662
    FNR                : 0.135802

  Metrics [Benign]:
    Accuracy   : 0.983891
    Precision  : 0.828133
    Recall     : 0.8

####**9.7 DNN - Binary OvR**

In [ ]:
DNN_UNITS      = [128, 64, 32, 16]
DNN_DROPOUT    = 0.3
DNN_LR         = 0.001
DNN_BATCH_SIZE = 8192  # Diperbesar dari 512 untuk mempercepat komputasi GPU pada 20M baris data
DNN_EPOCHS     = 50
DNN_PATIENCE   = 5     # Diperkecil dari 10 agar early stopping lebih cepat memotong epoch yang stagnan

print("Arsitektur DNN Binary OvR:")
print(f"  Hidden layers  : {DNN_UNITS}")
print(f"  Dropout        : {DNN_DROPOUT}")
print(f"  Learning rate  : {DNN_LR}")
print(f"  Batch size     : {DNN_BATCH_SIZE:,}")
print(f"  Max epochs     : {DNN_EPOCHS}")
print(f"  Early stop pat.: {DNN_PATIENCE}")
print(f"  class_weight   : None")

def build_dnn_binary(n_features, units, dropout, lr):
    """DNN untuk klasifikasi binary."""
    inputs = keras.Input(shape=(n_features,), name='input')
    x = inputs
    for i, u in enumerate(units):
        x = layers.Dense(u, name=f'dense_{i+1}')(x)
        x = layers.BatchNormalization(name=f'bn_{i+1}')(x)
        x = layers.Activation('relu', name=f'relu_{i+1}')(x)
        x = layers.Dropout(dropout, name=f'dropout_{i+1}')(x)
    # Output binary: sigmoid
    outputs = layers.Dense(1, activation='sigmoid', name='output')(x)
    model = keras.Model(inputs, outputs, name='DNN_Binary')
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=lr),
        loss='binary_crossentropy',
        metrics=['accuracy',
                 tf.keras.metrics.AUC(name='roc_auc'),
                 tf.keras.metrics.AUC(curve='PR', name='pr_auc')]
    )
    return model

dnn_results_per_class = {}
y_prob_dnn_matrix = np.zeros((len(y_test), N_CLASSES), dtype=np.float32)

t_dnn_total = 0
for enc in range(N_CLASSES):
    class_name = LABEL_NAMES[enc]
    print(f"\n--- DNN | Kelas {enc}: {class_name} ---")

    y_train_bin = (y_train_final == enc).astype(np.float32)
    y_test_bin  = (y_test == enc).astype(np.int32)

    n_pos_train = int(y_train_bin.sum())
    n_neg_train = int((y_train_bin == 0).sum())
    n_pos_test  = int(y_test_bin.sum())
    print(f"  Train: pos={n_pos_train:,}  neg={n_neg_train:,}")
    print(f"  Test : pos={n_pos_test:,}")

    # Build model
    tf.keras.backend.clear_session() # Bersihkan memory GPU/Keras
    tf.random.set_seed(42)
    dnn_bin = build_dnn_binary(N_FEATURES, DNN_UNITS, DNN_DROPOUT, DNN_LR)

    # Checkpoint per kelas
    path_ckpt = os.path.join(
        BINARY_DNN_DIR, f'dnn_bin_{class_name.lower()}_best.keras'
    )
    callbacks = [
        EarlyStopping(
            monitor='val_loss', patience=DNN_PATIENCE,
            restore_best_weights=True, verbose=1
        ),
        ReduceLROnPlateau(
            monitor='val_loss', factor=0.5,
            patience=3, min_lr=1e-6, verbose=1  # Kesabaran LR jg diturunkan menjadi 3
        ),
        ModelCheckpoint(
            filepath=path_ckpt, monitor='val_loss',
            save_best_only=True, verbose=0
        )
    ]

    t0 = time.time()
    history = dnn_bin.fit(
        X_train_final, y_train_bin,
        validation_data=(X_test, y_test_bin),
        batch_size=DNN_BATCH_SIZE,
        epochs=DNN_EPOCHS,
        callbacks=callbacks,
        verbose=0
    )
    t_elapsed = time.time() - t0
    t_dnn_total += t_elapsed

    best_epoch = int(np.argmin(history.history['val_loss']) + 1)
    print(f"  Training selesai: {t_elapsed:.2f}s")
    print(f"  Epoch terbaik   : {best_epoch}")

    # Prediksi
    y_prob_bin = dnn_bin.predict(
        X_test, batch_size=DNN_BATCH_SIZE, verbose=0
    ).flatten().astype(np.float32)
    y_pred_bin = (y_prob_bin >= 0.5).astype(np.int32)
    y_prob_dnn_matrix[:, enc] = y_prob_bin

    res = compute_binary_metrics(
        y_test_bin, y_pred_bin, y_prob_bin, class_name
    )
    res['training_time_sec'] = float(t_elapsed)
    res['best_epoch']        = best_epoch
    res['best_val_loss']     = float(min(history.history['val_loss']))
    dnn_results_per_class[class_name] = res

    # Save model dan history
    dnn_bin.save(
        os.path.join(BINARY_DNN_DIR, f'dnn_bin_{class_name.lower()}.keras')
    )
    history_dict = {k: [float(v) for v in vals]
                    for k, vals in history.history.items()}
    with open(os.path.join(
        BINARY_DNN_DIR, f'dnn_history_{class_name.lower()}.json'), 'w') as f:
        json.dump(history_dict, f, indent=4)

    del dnn_bin, y_train_bin, y_pred_bin, y_prob_bin, history
    gc.collect()

y_pred_dnn_ovr = np.argmax(y_prob_dnn_matrix, axis=1).astype(np.int32)
dnn_summary    = compute_ovr_summary(dnn_results_per_class, "DNN")

np.save(os.path.join(BINARY_DNN_DIR, 'y_pred_dnn_ovr.npy'), y_pred_dnn_ovr)
np.save(os.path.join(BINARY_DNN_DIR, 'y_prob_dnn_matrix.npy'), y_prob_dnn_matrix)

all_results_binary['DNN'] = {
    'per_class'         : dnn_results_per_class,
    'summary'           : dnn_summary,
    'total_training_sec': float(t_dnn_total)
}

with open(os.path.join(BINARY_RESULTS_DIR, 'result_dnn_binary.json'), 'w') as f:
    json.dump(all_results_binary['DNN'], f, indent=4)

print(f"\nTotal waktu DNN OvR : {t_dnn_total:.2f}s ({t_dnn_total/60:.2f} menit)")
print_ram("setelah DNN OvR")
del y_prob_dnn_matrix
gc.collect()
print("DNN Binary OvR selesai.")


Arsitektur DNN Binary OvR:
  Hidden layers  : [128, 64, 32, 16]
  Dropout        : 0.3
  Learning rate  : 0.001
  Batch size     : 8,192
  Max epochs     : 50
  Early stop pat.: 5
  class_weight   : None

--- DNN | Kelas 0: Benign ---
  Train: pos=1,000,000  neg=19,302,105
  Test : pos=218,760

Epoch 15: ReduceLROnPlateau reducing learning rate to 0.0005000000237487257.

Epoch 22: ReduceLROnPlateau reducing learning rate to 0.0002500000118743628.

Epoch 28: ReduceLROnPlateau reducing learning rate to 0.0001250000059371814.

Epoch 35: ReduceLROnPlateau reducing learning rate to 6.25000029685907e-05.

Epoch 38: ReduceLROnPlateau reducing learning rate to 3.125000148429535e-05.

Epoch 44: ReduceLROnPlateau reducing learning rate to 1.5625000742147677e-05.

Epoch 47: ReduceLROnPlateau reducing learning rate to 7.812500371073838e-06.

Epoch 50: ReduceLROnPlateau reducing learning rate to 3.906250185536919e-06.
Restoring model weights from the end of the best epoch: 48.
  Training selesai: 6

####**9.8 Save Semua Hasil dan Ringkasan**

In [ ]:
path_all = os.path.join(BINARY_RESULTS_DIR, 'training_results_binary.json')
with open(path_all, 'w') as f:
    json.dump(all_results_binary, f, indent=4)
print(f"Semua hasil tersimpan: {path_all}")

# Simpan combined predictions untuk Langkah 10
MODEL_NAMES = ['RandomForest', 'XGBoost', 'LightGBM', 'DNN']
pred_map = {}

if 'y_pred_rf_ovr' in globals():
    pred_map['RandomForest'] = y_pred_rf_ovr
if 'y_pred_xgb_ovr' in globals():
    pred_map['XGBoost'] = y_pred_xgb_ovr
if 'y_pred_lgb_ovr' in globals():
    pred_map['LightGBM'] = y_pred_lgb_ovr
if 'y_pred_dnn_ovr' in globals():
    pred_map['DNN'] = y_pred_dnn_ovr

for mname, y_pred in pred_map.items():
    key  = mname.lower().replace('randomforest', 'rf') \
                        .replace('xgboost', 'xgb') \
                        .replace('lightgbm', 'lgb')
    np.save(os.path.join(BINARY_RESULTS_DIR, f'y_pred_{key}_ovr.npy'), y_pred)

if 'y_test' in globals():
    np.save(os.path.join(BINARY_RESULTS_DIR, 'y_test.npy'), y_test)


Semua hasil tersimpan: /content/drive/My Drive/Framework/CICIoT2023/Binary/Results/training_results_binary.json


####**9.9 Ringkasan Akhir Langkah 9 Binary OvR**

In [ ]:
print(f"\n{'Classifier':15s} {'MacroP':>8s} {'MacroR':>8s} "
      f"{'MacroF1':>8s} {'MacroMCC':>9s} {'ROC-AUC':>8s} {'Time(m)':>8s}")
print("-" * 72)

for mname in MODEL_NAMES:
    if mname not in all_results_binary:
        continue
    s    = all_results_binary[mname]['summary']
    t_m  = all_results_binary[mname]['total_training_sec'] / 60
    roc  = s.get('macro_roc_auc') or 0.0
    print(f"  {mname:13s} "
          f"{s['macro_precision']:8.4f} "
          f"{s['macro_recall']:8.4f} "
          f"{s['macro_f1']:8.4f} "
          f"{s['macro_mcc']:9.4f} "
          f"{roc:8.4f} "
          f"{t_m:8.2f}")

print("-" * 72)

print(f"\nPer-class F1 (Binary OvR):")
print(f"  {'Kelas':15s}", end="")
for mname in MODEL_NAMES:
    if mname not in all_results_binary:
        continue
    short = mname.replace('RandomForest', 'RF') \
                 .replace('XGBoost', 'XGB') \
                 .replace('LightGBM', 'LGBM')
    print(f"  {short:>8s}", end="")
print()
print("  " + "-" * 52)

for cls in [LABEL_NAMES[i] for i in range(N_CLASSES)]:
    print(f"  {cls:15s}", end="")
    for mname in MODEL_NAMES:
        if mname not in all_results_binary:
            continue
        f1 = all_results_binary[mname]['per_class'] \
                 .get(cls, {}).get('f1', 0)
        print(f"  {f1:8.4f}", end="")
    print()

print(f"\nSemua file tersimpan di: {BINARY_RESULTS_DIR}")
print("\nLanjutkan ke Langkah 10 (Binary OvR): Evaluation & Comparison.")
print("=" * 60)


Classifier        MacroP   MacroR  MacroF1  MacroMCC  ROC-AUC  Time(m)
------------------------------------------------------------------------
  RandomForest    0.8990   0.6356   0.6850    0.6872   0.9795   156.99
  XGBoost         0.8794   0.6540   0.7018    0.6977   0.9798     5.23
  LightGBM        0.8716   0.6466   0.6929    0.6883   0.9787    11.86
  DNN             0.8692   0.6078   0.6504    0.6512   0.9763    76.22
------------------------------------------------------------------------

Per-class F1 (Binary OvR):
  Kelas                  RF       XGB      LGBM       DNN
  ----------------------------------------------------
  Benign             0.8513    0.8525    0.8458    0.8230
  Brute_Force        0.3498    0.3983    0.3808    0.2837
  DDoS               0.8949    0.8938    0.8912    0.8914
  DoS                0.5486    0.5725    0.5583    0.5242
  Mirai              0.9985    0.9986    0.9982    0.9982
  Recon              0.7614    0.7611    0.7545    0.7393
  Spoofin